In [1]:
import pandas as pd
# Set option to display all columns
pd.set_option('display.max_columns', None)


# import data

In [2]:
import duckdb
from pathlib import Path

con = duckdb.connect()

# Low-memory settings
con.execute("PRAGMA threads=1;")
con.execute("PRAGMA preserve_insertion_order=false;")
con.execute("PRAGMA enable_object_cache=false;")
con.execute("PRAGMA memory_limit='2GB';")           # try 1GB if still unstable
con.execute("PRAGMA temp_directory='data/tmp_duckdb';")

# 2) Build paths robustly from the notebook folder
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

BASE = ROOT / "data" / "by_server"

# IMPORTANT: your files are hive-partitioned like:
all_backends = (BASE / "*" / "*.parquet").as_posix()

con.execute(f"""
CREATE OR REPLACE VIEW all_backends AS
SELECT * FROM read_parquet('{all_backends}', hive_partitioning=true, union_by_name=true);
""")

# A unified "all_rows" view
con.execute("""
CREATE OR REPLACE VIEW all_rows AS
SELECT * FROM all_backends
""")

print(con.execute("SHOW TABLES").fetchall())


[('all_backends',), ('all_rows',)]


In [3]:
con.execute("""
SELECT backend, COUNT(*) AS total, COUNT(record_id) AS with_record_id
FROM (
    SELECT backend, record_id FROM all_backends
)
GROUP BY backend
""").df()


,backend,total,with_record_id
0,crossref,3919449,3919449
1,datacite,3525031,3525031
2,openalex,2352367,2352367
3,jxiv,902,902


In [4]:
con.execute(f"""
CREATE OR REPLACE VIEW server_thin AS
SELECT
  CAST(record_id AS VARCHAR)           AS record_id,
  CAST(server_name AS VARCHAR)         AS server_name,
  CAST(backend AS VARCHAR)             AS backend,

  CAST(doi AS VARCHAR)                 AS doi,
  CAST(doi_url AS VARCHAR)             AS doi_url,
  CAST(landing_page_url AS VARCHAR)    AS landing_page_url,
  
  -- Dates (helpful for temporal patterns)
  CAST(publication_year AS VARCHAR)    AS publication_year,
  CAST(date_created AS VARCHAR)        AS date_created,
  CAST(date_posted AS VARCHAR)         AS date_posted,
  CAST(date_deposited AS VARCHAR)      AS date_deposited,
  CAST(date_published AS VARCHAR)      AS date_published,
  CAST(date_published_online AS VARCHAR)      AS date_published_online,
  CAST(date_issued AS VARCHAR)         AS date_issued,
  CAST(date_indexed AS VARCHAR)        AS date_indexed,
  CAST(date_updated AS VARCHAR)        AS date_updated,
  CAST(date_registered AS VARCHAR)     AS date_registered,
  

FROM all_backends
""")

con.execute("SELECT COUNT(*) AS n FROM server_thin").df()


,n
0,9797749


In [5]:
data = con.execute("SELECT * FROM server_thin").df()
# data.drop_duplicates(subset=['record_id'], keep='first', inplace=False)

data = data.drop_duplicates()
data

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered
0,crossref::10.21467/preprints.48,AIJR Preprints,crossref,10.21467/preprints.48,https://doi.org/10.21467/preprints.48,https://preprints.aijr.org/index.php/ap/prepri...,2020.0,2020-09-15,2020-05-03,2020-09-15,2020-05-03,None,2020-05-03,2025-05-14,None,None
1,crossref::10.21467/preprints.43,AIJR Preprints,crossref,10.21467/preprints.43,https://doi.org/10.21467/preprints.43,https://preprints.aijr.org/index.php/ap/prepri...,2020.0,2020-09-15,2020-04-25,2020-09-15,2020-04-25,None,2020-04-25,2025-05-14,None,None
2,crossref::10.21467/preprints.39,AIJR Preprints,crossref,10.21467/preprints.39,https://doi.org/10.21467/preprints.39,https://preprints.aijr.org/index.php/ap/prepri...,2020.0,2020-09-15,2020-04-16,2020-09-15,2020-04-16,None,2020-04-16,2025-05-14,None,None
3,crossref::10.21467/preprints.38,AIJR Preprints,crossref,10.21467/preprints.38,https://doi.org/10.21467/preprints.38,https://preprints.aijr.org/index.php/ap/prepri...,2020.0,2020-09-17,2020-04-15,2020-09-17,2020-04-15,None,2020-04-15,2022-12-13,None,None
4,crossref::10.21467/preprints.36,AIJR Preprints,crossref,10.21467/preprints.36,https://doi.org/10.21467/preprints.36,https://preprints.aijr.org/index.php/ap/prepri...,2020.0,2020-09-17,2020-04-15,2020-09-17,2020-04-15,None,2020-04-15,2024-08-11,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9797744,openalex::W999325625,viXra,openalex,None,None,https://vixra.org/pdf/1409.0090v1.pdf,2014.0,2025-10-10T00:00:00,None,None,2014-09-01,None,None,None,2025-10-10T17:16:08.811792,None
9797745,openalex::W999460032,viXra,openalex,None,None,https://vixra.org/abs/1112.0094,2011.0,2025-10-10T00:00:00,None,None,2011-12-01,None,None,None,2025-10-10T17:16:08.811792,None
9797746,openalex::W99967155,viXra,openalex,None,None,https://vixra.org/pdf/1406.0019v1.pdf,2014.0,2025-10-10T00:00:00,None,None,2014-06-01,None,None,None,2025-10-10T17:16:08.811792,None
9797747,openalex::W999790414,viXra,openalex,None,None,https://vixra.org/pdf/1306.0105v3.pdf,2013.0,2025-10-10T00:00:00,None,None,2013-06-01,None,None,None,2025-10-10T17:16:08.811792,None


# "dateType": "Submitted"

In [6]:
import pandas as pd
# ROOT = '/mnt/c/SCHOLCOMMLAB/APPs/preprint-harvester'
date_submit_df = pd.read_parquet(f"{ROOT}/data/submitted_dates.parquet")
print(date_submit_df.shape)
date_submit_df.head()

(3524938, 6)


,record_id,server_name,backend,date_submitted,date_first,raw_dates
0,datacite::10.11588/artdok.00008094,ART-Dok,datacite,None,2023,"[{""date"": ""2023"", ""dateType"": ""Issued""}]"
1,datacite::10.11588/artdok.00008100,ART-Dok,datacite,None,2023,"[{""date"": ""2023"", ""dateType"": ""Issued""}]"
2,datacite::10.11588/artdok.00008101,ART-Dok,datacite,None,2023,"[{""date"": ""2023"", ""dateType"": ""Issued""}]"
3,datacite::10.11588/artdok.00008102,ART-Dok,datacite,None,2023,"[{""date"": ""2023"", ""dateType"": ""Issued""}]"
4,datacite::10.11588/artdok.00008103,ART-Dok,datacite,None,2023,"[{""date"": ""2023"", ""dateType"": ""Issued""}]"


In [7]:
date_submit_df[date_submit_df['server_name']=='Arabixiv']

,record_id,server_name,backend,date_submitted,date_first,raw_dates


In [8]:
pattern = r'"dateType": "Submitted"'
# pattern = r'v\d+$'
mask = ~date_submit_df['raw_dates'].str.contains(pattern, regex=True, na=False)
result = date_submit_df[mask]
result

,record_id,server_name,backend,date_submitted,date_first,raw_dates
0,datacite::10.11588/artdok.00008094,ART-Dok,datacite,None,2023,"[{""date"": ""2023"", ""dateType"": ""Issued""}]"
1,datacite::10.11588/artdok.00008100,ART-Dok,datacite,None,2023,"[{""date"": ""2023"", ""dateType"": ""Issued""}]"
2,datacite::10.11588/artdok.00008101,ART-Dok,datacite,None,2023,"[{""date"": ""2023"", ""dateType"": ""Issued""}]"
3,datacite::10.11588/artdok.00008102,ART-Dok,datacite,None,2023,"[{""date"": ""2023"", ""dateType"": ""Issued""}]"
4,datacite::10.11588/artdok.00008103,ART-Dok,datacite,None,2023,"[{""date"": ""2023"", ""dateType"": ""Issued""}]"
...,...,...,...,...,...,...
3524933,datacite::10.17605/osf.io/rkzs2,Open Science Framework,datacite,None,2023-04-25,"[{""date"": ""2023-04-25"", ""dateType"": ""Created""}..."
3524934,datacite::10.25926/9bys-5148,ELPUB (Universitat Wuppertal),datacite,None,2024,"[{""date"": ""2024"", ""dateInformation"": null, ""da..."
3524935,datacite::10.25926/x4af-1639,ELPUB (Universitat Wuppertal),datacite,None,2024,"[{""date"": ""2024"", ""dateInformation"": null, ""da..."
3524936,datacite::10.25926/c7nb-8j29,ELPUB (Universitat Wuppertal),datacite,None,2024,"[{""date"": ""2024"", ""dateType"": ""Issued""}]"


In [9]:
import duckdb

con = duckdb.connect()

# ── Register DataFrames as virtual tables ─────────────────────────────────────
con.register("submitted", date_submit_df)
con.register("main_data", data)

# ── LEFT JOIN: keep all rows from `data` ─────────────────────────────────────
merged = con.execute("""
    SELECT
        d.*,
        s.date_submitted,
        s.date_first, 
        s.raw_dates
    FROM main_data AS d
    LEFT JOIN submitted AS s
        ON  d.record_id   = s.record_id
        AND d.server_name = s.server_name
        AND d.backend     = s.backend
""").df()

print(merged.shape)
merged.head()

(8031092, 19)


,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates
0,crossref::10.32942/osf.io/brfas,EcoEvoRxiv,crossref,10.32942/osf.io/brfas,https://doi.org/10.32942/osf.io/brfas,https://ecoevorxiv.org/repository/view/3707,2022.0,2022-07-18,2022-07-17,2022-10-25,2022-07-17,None,2022-07-17,2025-10-26,None,None,None,None,None
1,crossref::10.32942/osf.io/gx65w,EcoEvoRxiv,crossref,10.32942/osf.io/gx65w,https://doi.org/10.32942/osf.io/gx65w,https://ecoevorxiv.org/repository/view/3708,2022.0,2022-07-16,2022-07-15,2022-10-25,2022-07-15,None,2022-07-15,2025-02-21,None,None,None,None,None
2,crossref::10.32942/osf.io/b6rf4,EcoEvoRxiv,crossref,10.32942/osf.io/b6rf4,https://doi.org/10.32942/osf.io/b6rf4,https://ecoevorxiv.org/repository/view/3709,2022.0,2022-07-14,2022-07-14,2022-10-25,2022-07-14,None,2022-07-14,2025-10-22,None,None,None,None,None
3,crossref::10.32942/osf.io/fb8k7,EcoEvoRxiv,crossref,10.32942/osf.io/fb8k7,https://doi.org/10.32942/osf.io/fb8k7,https://ecoevorxiv.org/repository/view/3710,2022.0,2022-07-13,2022-07-13,2022-10-25,2022-07-13,None,2022-07-13,2025-02-21,None,None,None,None,None
4,crossref::10.32942/osf.io/sm6xq,EcoEvoRxiv,crossref,10.32942/osf.io/sm6xq,https://doi.org/10.32942/osf.io/sm6xq,https://ecoevorxiv.org/repository/view/3711,2022.0,2022-07-11,2022-07-11,2022-10-25,2022-07-11,None,2022-07-11,2022-10-26,None,None,None,None,None


# Compute the earliest known date for each record and extract its year

In [10]:
import pandas as pd
import numpy as np

def safe_parse_date(series: pd.Series) -> pd.Series:
    """
    Robustly parse a mixed-format date Series to datetime.
    - YYYY           → NaT  (too imprecise, avoid false early dates)
    - YYYY-MM        → NaT  (same reason)
    - YYYY-MM-DD     → parsed normally
    - ISO timestamps → parsed normally
    """
    s = series.astype(str).str.strip()
    s = s.replace({"None": np.nan, "nan": np.nan, "NaT": np.nan, "": np.nan})

    # ── Mask imprecise values BEFORE parsing ──────────────────────────────────
    # .fillna(False) prevents NaN → float issue when using ~ operator
    is_year_only       = s.str.match(r"^\d{4}$").fillna(False)
    is_year_month_only = s.str.match(r"^\d{4}-\d{2}$").fillna(False)
    s = s.where(~is_year_only & ~is_year_month_only, other=np.nan)

    # ── Normalize timezone suffix ─────────────────────────────────────────────
    s = s.str.replace(r"Z$", "+00:00", regex=True)

    return pd.to_datetime(s, errors="coerce", utc=False)


def compute_earliest_date_and_year(
    df: pd.DataFrame,
    date_cols,
    date_col_out: str = "date_first_seen",
    year_col_out: str = "publication_year_first_seen",
) -> pd.DataFrame:

    df = df.copy()
    existing_cols = [c for c in date_cols if c in df.columns]

    if not existing_cols:
        df[date_col_out] = pd.NaT
        df[year_col_out] = pd.NA
        df[year_col_out] = df[year_col_out].astype("Int64")
        return df

    parsed_dates = pd.DataFrame(
        {col: safe_parse_date(df[col]) for col in existing_cols},
        index=df.index,
    )

    df[date_col_out] = parsed_dates.min(axis=1)
    df[year_col_out] = df[date_col_out].dt.year.astype("Int64")

    return df


# ── Run ───────────────────────────────────────────────────────────────────────
DATE_COLUMNS = [
    "date_created",
    "date_posted",
    "date_deposited",
    "date_published",
    "date_published_online",
    "date_issued",
    "date_indexed",
    "date_updated",
    "date_registered",
    "date_first",
    "date_submitted",
]

data_date_first_seen = compute_earliest_date_and_year(
    merged,
    DATE_COLUMNS,
    date_col_out="date_first_seen",
    year_col_out="publication_year_first_seen",
)

# ── Sanity check ──────────────────────────────────────────────────────────────
print("Shape:", data_date_first_seen.shape)
print("\nNull rate per date column:")
print(data_date_first_seen[DATE_COLUMNS].isna().mean().round(3).to_string())
print("\ndate_first_seen nulls:", data_date_first_seen["date_first_seen"].isna().sum())
print("year range:",
      data_date_first_seen["publication_year_first_seen"].min(), "→",
      data_date_first_seen["publication_year_first_seen"].max())

data_date_first_seen[["record_id", "date_first_seen", "publication_year_first_seen"]].head(10)

/tmp/ipykernel_39206/3297356985.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  is_year_only       = s.str.match(r"^\d{4}$").fillna(False)
/tmp/ipykernel_39206/3297356985.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  is_year_month_only = s.str.match(r"^\d{4}-\d{2}$").fillna(False)
/tmp/ipykernel_39206/3297356985.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('f

Shape: (8031092, 21)

Null rate per date column:
date_created             0.000
date_posted              0.757
date_deposited           0.651
date_published           0.439
date_published_online    0.977
date_issued              0.652
date_indexed             0.651
date_updated             0.349
date_registered          0.561
date_first               0.561
date_submitted           0.635

date_first_seen nulls: 0
year range: 1828 → 2026


,record_id,date_first_seen,publication_year_first_seen
0,crossref::10.32942/osf.io/brfas,2022-07-17,2022
1,crossref::10.32942/osf.io/gx65w,2022-07-15,2022
2,crossref::10.32942/osf.io/b6rf4,2022-07-14,2022
3,crossref::10.32942/osf.io/fb8k7,2022-07-13,2022
4,crossref::10.32942/osf.io/sm6xq,2022-07-11,2022
5,crossref::10.32942/osf.io/u86q7,2022-07-05,2022
6,crossref::10.32942/osf.io/642xb,2022-07-05,2022
7,crossref::10.32942/osf.io/2f6uk,2022-07-04,2022
8,crossref::10.32942/osf.io/8mgv6,2022-07-01,2022
9,crossref::10.32942/osf.io/hya68,2022-07-03,2022


# matching ssrn dates

In [11]:
data_date_first_seen_ssrn = data_date_first_seen[data_date_first_seen['server_name']=='SSRN']
data_date_first_seen_ssrn

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen
2247485,crossref::10.2139/ssrn.1912358,SSRN,crossref,10.2139/ssrn.1912358,https://doi.org/10.2139/ssrn.1912358,http://www.ssrn.com/abstract=1912358,2011.0,2012-01-05,None,2019-05-01,2011-01-01,None,2011-01-01,2022-04-03,None,None,None,None,None,2011-01-01,2011
2247486,crossref::10.2139/ssrn.2448644,SSRN,crossref,10.2139/ssrn.2448644,https://doi.org/10.2139/ssrn.2448644,http://www.ssrn.com/abstract=2448644,2013.0,2014-06-18,None,2019-05-01,2013-01-01,None,2013-01-01,2022-04-01,None,None,None,None,None,2013-01-01,2013
2247487,crossref::10.2139/ssrn.1703883,SSRN,crossref,10.2139/ssrn.1703883,https://doi.org/10.2139/ssrn.1703883,http://www.ssrn.com/abstract=1703883,2010.0,2012-01-05,None,2019-05-01,2010-01-01,None,2010-01-01,2022-03-29,None,None,None,None,None,2010-01-01,2010
2247488,crossref::10.2139/ssrn.796126,SSRN,crossref,10.2139/ssrn.796126,https://doi.org/10.2139/ssrn.796126,http://www.ssrn.com/abstract=796126,2005.0,2011-12-28,None,2019-05-01,2005-01-01,None,2005-01-01,2022-04-01,None,None,None,None,None,2005-01-01,2005
2247489,crossref::10.2139/ssrn.2484022,SSRN,crossref,10.2139/ssrn.2484022,https://doi.org/10.2139/ssrn.2484022,http://www.ssrn.com/abstract=2484022,2014.0,2014-08-27,None,2019-05-01,2014-01-01,None,2014-01-01,2022-04-04,None,None,None,None,None,2014-01-01,2014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5776184,crossref::10.2139/ssrn.5366340,SSRN,crossref,10.2139/ssrn.5366340,https://doi.org/10.2139/ssrn.5366340,https://www.ssrn.com/abstract=5366340,2025.0,2025-08-01,2025-01-01,2025-08-01,2025-01-01,None,2025-01-01,2025-08-01,None,None,None,None,None,2025-01-01,2025
5776185,crossref::10.2139/ssrn.5376066,SSRN,crossref,10.2139/ssrn.5376066,https://doi.org/10.2139/ssrn.5376066,https://www.ssrn.com/abstract=5376066,2025.0,2025-08-01,2025-01-01,2025-08-01,2025-01-01,None,2025-01-01,2025-08-01,None,None,None,None,None,2025-01-01,2025
5776186,crossref::10.2139/ssrn.5376068,SSRN,crossref,10.2139/ssrn.5376068,https://doi.org/10.2139/ssrn.5376068,https://www.ssrn.com/abstract=5376068,2025.0,2025-08-01,2025-01-01,2025-08-01,2025-01-01,None,2025-01-01,2025-08-01,None,None,None,None,None,2025-01-01,2025
5776187,crossref::10.2139/ssrn.5376087,SSRN,crossref,10.2139/ssrn.5376087,https://doi.org/10.2139/ssrn.5376087,https://www.ssrn.com/abstract=5376087,2025.0,2025-08-01,2025-01-01,2025-08-01,2025-01-01,None,2025-01-01,2025-08-01,None,None,None,None,None,2025-01-01,2025


In [12]:
data_date_first_seen_ssrn.count()

record_id                      1303005
server_name                    1303005
backend                        1303005
doi                            1303005
doi_url                        1303005
landing_page_url               1303005
publication_year               1303005
date_created                   1303005
date_posted                     515391
date_deposited                 1303005
date_published                 1303005
date_published_online           125442
date_issued                    1303005
date_indexed                   1303005
date_updated                         0
date_registered                      0
date_submitted                       0
date_first                           0
raw_dates                            0
date_first_seen                1303005
publication_year_first_seen    1303005
dtype: int64

In [13]:
df_before1990 = data_date_first_seen[data_date_first_seen['publication_year_first_seen'] < 1990]
df_before1990

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen
891783,crossref::10.20948/prepr-1981-43,Keldysh Institute Preprints,crossref,10.20948/prepr-1981-43,https://doi.org/10.20948/prepr-1981-43,http://keldysh.ru/papers/1981/prep1981_43.pdf,1981.0,2021-07-22,None,2021-07-22,1981-01-01,1981-01-01,1981-01-01,2025-02-21,None,None,None,None,None,1981-01-01,1981
891784,crossref::10.20948/prepr-1984-102,Keldysh Institute Preprints,crossref,10.20948/prepr-1984-102,https://doi.org/10.20948/prepr-1984-102,http://keldysh.ru/papers/1984/prep1984_102.pdf,1984.0,2021-07-22,None,2021-07-22,1984-01-01,1984-01-01,1984-01-01,2025-02-21,None,None,None,None,None,1984-01-01,1984
2248136,crossref::10.2139/ssrn.1370672,SSRN,crossref,10.2139/ssrn.1370672,https://doi.org/10.2139/ssrn.1370672,http://www.ssrn.com/abstract=1370672,1984.0,2011-12-28,None,2019-05-01,1984-01-01,None,1984-01-01,2022-03-30,None,None,None,None,None,1984-01-01,1984
2248676,crossref::10.2139/ssrn.1645431,SSRN,crossref,10.2139/ssrn.1645431,https://doi.org/10.2139/ssrn.1645431,http://www.ssrn.com/abstract=1645431,1987.0,2012-01-04,None,2019-05-01,1987-01-01,None,1987-01-01,2022-04-04,None,None,None,None,None,1987-01-01,1987
2249134,crossref::10.2139/ssrn.1644966,SSRN,crossref,10.2139/ssrn.1644966,https://doi.org/10.2139/ssrn.1644966,http://www.ssrn.com/abstract=1644966,1982.0,2012-01-04,None,2019-05-01,1982-01-01,None,1982-01-01,2022-04-04,None,None,None,None,None,1982-01-01,1982
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6545706,datacite::10.48550/arxiv.math/9201206,arXiv,datacite,10.48550/arxiv.math/9201206,https://doi.org/10.48550/arxiv.math/9201206,https://arxiv.org/abs/math/9201206,1992.0,2022-03-20,None,None,None,None,None,None,2022-03-20,2022-03-20,1989-11-09,1989-11-09,"[{""date"": ""1989-11-09T15:37:00Z"", ""dateInforma...",1989-11-09,1989
7543346,datacite::10.48550/arxiv.math/9201207,arXiv,datacite,10.48550/arxiv.math/9201207,https://doi.org/10.48550/arxiv.math/9201207,https://arxiv.org/abs/math/9201207,1992.0,2022-03-20,None,None,None,None,None,None,2022-03-20,2022-03-20,1989-11-17,1989-11-17,"[{""date"": ""1989-11-17T15:28:00Z"", ""dateInforma...",1989-11-17,1989
7543411,datacite::10.48550/arxiv.math/9201239,arXiv,datacite,10.48550/arxiv.math/9201239,https://doi.org/10.48550/arxiv.math/9201239,https://arxiv.org/abs/math/9201239,1992.0,2022-03-20,None,None,None,None,None,None,2022-03-20,2022-03-20,1989-04-15,1989-04-15,"[{""date"": ""1989-04-15T00:00:00Z"", ""dateInforma...",1989-04-15,1989
7543502,datacite::10.48550/arxiv.hep-th/9108028,arXiv,datacite,10.48550/arxiv.hep-th/9108028,https://doi.org/10.48550/arxiv.hep-th/9108028,https://arxiv.org/abs/hep-th/9108028,1991.0,2022-03-20,None,None,None,None,None,None,2022-03-20,2022-03-20,1988-11-11,1988-11-11,"[{""date"": ""1988-11-11T15:39:49Z"", ""dateInforma...",1988-11-11,1988


In [14]:
df_before1990[df_before1990['server_name']=='SSRN']

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen
2248136,crossref::10.2139/ssrn.1370672,SSRN,crossref,10.2139/ssrn.1370672,https://doi.org/10.2139/ssrn.1370672,http://www.ssrn.com/abstract=1370672,1984.0,2011-12-28,None,2019-05-01,1984-01-01,None,1984-01-01,2022-03-30,None,None,None,None,None,1984-01-01,1984
2248676,crossref::10.2139/ssrn.1645431,SSRN,crossref,10.2139/ssrn.1645431,https://doi.org/10.2139/ssrn.1645431,http://www.ssrn.com/abstract=1645431,1987.0,2012-01-04,None,2019-05-01,1987-01-01,None,1987-01-01,2022-04-04,None,None,None,None,None,1987-01-01,1987
2249134,crossref::10.2139/ssrn.1644966,SSRN,crossref,10.2139/ssrn.1644966,https://doi.org/10.2139/ssrn.1644966,http://www.ssrn.com/abstract=1644966,1982.0,2012-01-04,None,2019-05-01,1982-01-01,None,1982-01-01,2022-04-04,None,None,None,None,None,1982-01-01,1982
2249448,crossref::10.2139/ssrn.1367829,SSRN,crossref,10.2139/ssrn.1367829,https://doi.org/10.2139/ssrn.1367829,http://www.ssrn.com/abstract=1367829,1982.0,2011-12-28,None,2019-05-01,1982-01-01,None,1982-01-01,2022-03-29,None,None,None,None,None,1982-01-01,1982
2270245,crossref::10.2139/ssrn.2807432,SSRN,crossref,10.2139/ssrn.2807432,https://doi.org/10.2139/ssrn.2807432,https://www.ssrn.com/abstract=2807432,1987.0,2017-06-28,None,2019-05-12,1987-01-01,None,1987-01-01,2022-04-04,None,None,None,None,None,1987-01-01,1987
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5009042,crossref::10.2139/ssrn.2116567,SSRN,crossref,10.2139/ssrn.2116567,https://doi.org/10.2139/ssrn.2116567,http://www.ssrn.com/abstract=2116567,1979.0,2012-08-29,None,2019-05-01,1979-01-01,None,1979-01-01,2022-04-02,None,None,None,None,None,1979-01-01,1979
5009054,crossref::10.2139/ssrn.2065413,SSRN,crossref,10.2139/ssrn.2065413,https://doi.org/10.2139/ssrn.2065413,http://www.ssrn.com/abstract=2065413,1982.0,2012-07-06,None,2019-05-01,1982-01-01,None,1982-01-01,2025-04-20,None,None,None,None,None,1982-01-01,1982
5009438,crossref::10.2139/ssrn.2118848,SSRN,crossref,10.2139/ssrn.2118848,https://doi.org/10.2139/ssrn.2118848,http://www.ssrn.com/abstract=2118848,1974.0,2012-08-29,None,2019-05-01,1974-01-01,None,1974-01-01,2022-04-03,None,None,None,None,None,1974-01-01,1974
5009532,crossref::10.2139/ssrn.2340845,SSRN,crossref,10.2139/ssrn.2340845,https://doi.org/10.2139/ssrn.2340845,http://www.ssrn.com/abstract=2340845,1975.0,2013-10-23,None,2019-05-01,1975-01-01,None,1975-01-01,2022-04-04,None,None,None,None,None,1975-01-01,1975


In [15]:
df_before1990['server_name'].value_counts()

server_name
SSRN                           686
Zenodo                          17
arXiv                            8
Keldysh Institute Preprints      2
Name: count, dtype: int64

In [16]:
df_before1990.columns

Index(['record_id', 'server_name', 'backend', 'doi', 'doi_url',
       'landing_page_url', 'publication_year', 'date_created', 'date_posted',
       'date_deposited', 'date_published', 'date_published_online',
       'date_issued', 'date_indexed', 'date_updated', 'date_registered',
       'date_submitted', 'date_first', 'raw_dates', 'date_first_seen',
       'publication_year_first_seen'],
      dtype='object')

## import SSRNData

In [17]:
import pandas as pd

# The 'r' before the string tells Python to treat backslashes as literal characters
file_path = r"/mnt/c/SCHOLCOMMLAB/APPS/preprint-harvester/data/SSRNData/SSRNData.txt"

# If the data is separated by tabs, use sep='\t'. If it's commas, you can remove the sep argument.
SSRNData = pd.read_csv(file_path, sep='\t') 

# Display the first 5 rows to make sure it loaded correctly
SSRNData.head()

,rank,url,title,authors,datePosted
0,1,https://papers.ssrn.com/sol3/papers.cfm?abstra...,Monetary Tightening and U.S. Bank Fragility in...,"<a href=""https://papers.ssrn.com/sol3/cf_dev/A...",24 Mar 2023
1,2,https://papers.ssrn.com/sol3/papers.cfm?abstra...,Why Do People Migrate? A Review of the Theoret...,"<a href=""https://papers.ssrn.com/sol3/cf_dev/A...",14 Mar 2008
2,3,https://papers.ssrn.com/sol3/papers.cfm?abstra...,The Sweep and Force of Section Three,"<a href=""https://papers.ssrn.com/sol3/cf_dev/A...",14 Aug 2023
3,4,https://papers.ssrn.com/sol3/papers.cfm?abstra...,Navigating the Jagged Technological Frontier: ...,"<a href=""https://papers.ssrn.com/sol3/cf_dev/A...",18 Sep 2023
4,5,https://papers.ssrn.com/sol3/papers.cfm?abstra...,Can ChatGPT Forecast Stock Price Movements? Re...,"<a href=""https://papers.ssrn.com/sol3/cf_dev/A...",10 Apr 2023


In [18]:
SSRNData.count()

rank          1137036
url           1137036
title         1137032
authors       1137036
datePosted    1137015
dtype: int64

In [19]:
SSRNData.url.unique()

array(['https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4387676',
       'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1105657',
       'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=4532751', ...,
       'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3655838',
       'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3655866',
       'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3647635'],
      shape=(1137036,), dtype=object)

In [20]:
SSRNData.authors[2]

'<a href="https://papers.ssrn.com/sol3/cf_dev/AbsByAuth.cfm?per_id=398074" target="_blank">William Baude</a> and <a href="https://papers.ssrn.com/sol3/cf_dev/AbsByAuth.cfm?per_id=967471" target="_blank">Michael Stokes Paulsen</a><br>University of Chicago - Law School and University of St. Thomas School of Law<br>'

In [21]:
import pandas as pd
import re

SSRNData_clean = SSRNData.copy()

# extract abstract_id
SSRNData_clean["ssrn_id"] = SSRNData_clean["url"].str.extract(r"abstract_id=(\d+)")

data_date_first_seen["ssrn_id"] = data_date_first_seen["doi"].str.extract(r"10.2139/ssrn.(\d+)")

# build DOI
SSRNData_clean["doi"] = "10.2139/ssrn." + SSRNData_clean["ssrn_id"]

# build DOI
SSRNData_clean["landing_page_url"] = "https://www.ssrn.com/abstract=" + SSRNData_clean["ssrn_id"]

# parse date
SSRNData_clean["date_posted_ssrn"] = pd.to_datetime(SSRNData_clean["datePosted"], errors="coerce")

SSRNData_clean["publication_year_ssrn"] = SSRNData_clean["date_posted_ssrn"].dt.year

# SSRNData_clean["landing_page_url"] = df["url"]

# SSRNData_clean["server_name"] = "SSRN"
# SSRNData_clean["backend"] = "SSRN"

SSRNData_clean.head()

,rank,url,title,authors,datePosted,ssrn_id,doi,landing_page_url,date_posted_ssrn,publication_year_ssrn
0,1,https://papers.ssrn.com/sol3/papers.cfm?abstra...,Monetary Tightening and U.S. Bank Fragility in...,"<a href=""https://papers.ssrn.com/sol3/cf_dev/A...",24 Mar 2023,4387676,10.2139/ssrn.4387676,https://www.ssrn.com/abstract=4387676,2023-03-24,2023.0
1,2,https://papers.ssrn.com/sol3/papers.cfm?abstra...,Why Do People Migrate? A Review of the Theoret...,"<a href=""https://papers.ssrn.com/sol3/cf_dev/A...",14 Mar 2008,1105657,10.2139/ssrn.1105657,https://www.ssrn.com/abstract=1105657,2008-03-14,2008.0
2,3,https://papers.ssrn.com/sol3/papers.cfm?abstra...,The Sweep and Force of Section Three,"<a href=""https://papers.ssrn.com/sol3/cf_dev/A...",14 Aug 2023,4532751,10.2139/ssrn.4532751,https://www.ssrn.com/abstract=4532751,2023-08-14,2023.0
3,4,https://papers.ssrn.com/sol3/papers.cfm?abstra...,Navigating the Jagged Technological Frontier: ...,"<a href=""https://papers.ssrn.com/sol3/cf_dev/A...",18 Sep 2023,4573321,10.2139/ssrn.4573321,https://www.ssrn.com/abstract=4573321,2023-09-18,2023.0
4,5,https://papers.ssrn.com/sol3/papers.cfm?abstra...,Can ChatGPT Forecast Stock Price Movements? Re...,"<a href=""https://papers.ssrn.com/sol3/cf_dev/A...",10 Apr 2023,4412788,10.2139/ssrn.4412788,https://www.ssrn.com/abstract=4412788,2023-04-10,2023.0


In [22]:
pattern = "10.2139/ssrn.4387676"

mask = data_date_first_seen['doi'].str.contains(pattern, regex=False, na=False)
result = data_date_first_seen[mask]
result

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id
4263493,crossref::10.2139/ssrn.4387676,SSRN,crossref,10.2139/ssrn.4387676,https://doi.org/10.2139/ssrn.4387676,https://www.ssrn.com/abstract=4387676,2023.0,2023-03-24,None,2024-07-13,2023-01-01,2023-01-01,2023-01-01,2026-04-04,None,None,None,None,None,2023-01-01,2023,4387676
4489180,crossref::10.2139/ssrn.4387676,SSRN,crossref,10.2139/ssrn.4387676,https://doi.org/10.2139/ssrn.4387676,https://www.ssrn.com/abstract=4387676,2023.0,2023-03-24,None,2024-07-13,2023-01-01,2023-01-01,2023-01-01,2025-12-13,None,None,None,None,None,2023-01-01,2023,4387676


In [23]:
SSRNData_clean[['doi','date_posted_ssrn','publication_year_ssrn']]

,doi,date_posted_ssrn,publication_year_ssrn
0,10.2139/ssrn.4387676,2023-03-24,2023.0
1,10.2139/ssrn.1105657,2008-03-14,2008.0
2,10.2139/ssrn.4532751,2023-08-14,2023.0
3,10.2139/ssrn.4573321,2023-09-18,2023.0
4,10.2139/ssrn.4412788,2023-04-10,2023.0
...,...,...,...
1137031,10.2139/ssrn.3655672,2020-08-24,2020.0
1137032,10.2139/ssrn.3655759,2020-07-30,2020.0
1137033,10.2139/ssrn.3655838,2020-08-25,2020.0
1137034,10.2139/ssrn.3655866,2020-07-31,2020.0


## merge ssrn data to others data

In [24]:
ssrn_map_doi = SSRNData_clean.set_index("doi")["date_posted_ssrn"]
ssrn_map_url = SSRNData_clean.set_index("landing_page_url")["date_posted_ssrn"]
ssrn_map_id = SSRNData_clean.set_index("ssrn_id")["date_posted_ssrn"]

ssrn_map_year = SSRNData_clean.set_index("doi")["publication_year_ssrn"]

data_date_first_seen["date_posted_ssrn_doi"] = data_date_first_seen["doi"].map(ssrn_map_doi)
data_date_first_seen["date_posted_ssrn_url"] = data_date_first_seen["landing_page_url"].map(ssrn_map_url)
data_date_first_seen["date_posted_ssrn_id"] = data_date_first_seen["ssrn_id"].map(ssrn_map_id)
data_date_first_seen["publication_year_ssrn"] = data_date_first_seen["doi"].map(ssrn_map_year)

In [25]:
data_date_first_seen

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn
0,crossref::10.32942/osf.io/brfas,EcoEvoRxiv,crossref,10.32942/osf.io/brfas,https://doi.org/10.32942/osf.io/brfas,https://ecoevorxiv.org/repository/view/3707,2022.0,2022-07-18,2022-07-17,2022-10-25,2022-07-17,None,2022-07-17,2025-10-26,None,None,None,None,None,2022-07-17,2022,NaN,NaT,NaT,NaT,NaN
1,crossref::10.32942/osf.io/gx65w,EcoEvoRxiv,crossref,10.32942/osf.io/gx65w,https://doi.org/10.32942/osf.io/gx65w,https://ecoevorxiv.org/repository/view/3708,2022.0,2022-07-16,2022-07-15,2022-10-25,2022-07-15,None,2022-07-15,2025-02-21,None,None,None,None,None,2022-07-15,2022,NaN,NaT,NaT,NaT,NaN
2,crossref::10.32942/osf.io/b6rf4,EcoEvoRxiv,crossref,10.32942/osf.io/b6rf4,https://doi.org/10.32942/osf.io/b6rf4,https://ecoevorxiv.org/repository/view/3709,2022.0,2022-07-14,2022-07-14,2022-10-25,2022-07-14,None,2022-07-14,2025-10-22,None,None,None,None,None,2022-07-14,2022,NaN,NaT,NaT,NaT,NaN
3,crossref::10.32942/osf.io/fb8k7,EcoEvoRxiv,crossref,10.32942/osf.io/fb8k7,https://doi.org/10.32942/osf.io/fb8k7,https://ecoevorxiv.org/repository/view/3710,2022.0,2022-07-13,2022-07-13,2022-10-25,2022-07-13,None,2022-07-13,2025-02-21,None,None,None,None,None,2022-07-13,2022,NaN,NaT,NaT,NaT,NaN
4,crossref::10.32942/osf.io/sm6xq,EcoEvoRxiv,crossref,10.32942/osf.io/sm6xq,https://doi.org/10.32942/osf.io/sm6xq,https://ecoevorxiv.org/repository/view/3711,2022.0,2022-07-11,2022-07-11,2022-10-25,2022-07-11,None,2022-07-11,2022-10-26,None,None,None,None,None,2022-07-11,2022,NaN,NaT,NaT,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8031087,datacite::10.48550/arxiv.2501.12237,arXiv,datacite,10.48550/arxiv.2501.12237,https://doi.org/10.48550/arxiv.2501.12237,https://arxiv.org/abs/2501.12237,2025.0,2025-01-22,None,None,None,None,None,None,2025-04-11,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T15:59:03Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN
8031088,datacite::10.48550/arxiv.2501.12238,arXiv,datacite,10.48550/arxiv.2501.12238,https://doi.org/10.48550/arxiv.2501.12238,https://arxiv.org/abs/2501.12238,2025.0,2025-01-22,None,None,None,None,None,None,2025-02-27,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T15:59:19Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN
8031089,datacite::10.48550/arxiv.2501.12248,arXiv,datacite,10.48550/arxiv.2501.12248,https://doi.org/10.48550/arxiv.2501.12248,https://arxiv.org/abs/2501.12248,2025.0,2025-01-22,None,None,None,None,None,None,2025-05-08,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T16:09:02Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN
8031090,datacite::10.48550/arxiv.2501.12281,arXiv,datacite,10.48550/arxiv.2501.12281,https://doi.org/10.48550/arxiv.2501.12281,https://arxiv.org/abs/2501.12281,2025.0,2025-01-22,None,None,None,None,None,None,2025-03-21,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T16:52:42Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN


In [26]:
date_posted_ssrn_doi = data_date_first_seen[data_date_first_seen["date_posted_ssrn_doi"].notna()]
date_posted_ssrn_doi

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn
7863,openalex::W3123789860,EconStor Preprints,openalex,10.2139/ssrn.2573596,https://doi.org/10.2139/ssrn.2573596,http://hdl.handle.net/10419/107595,2015.0,2025-10-10T00:00:00,None,None,2015-01-01,None,None,None,2025-10-10T17:16:08.811792,None,None,None,None,2015-01-01,2015,2573596,2015-03-05,NaT,2015-03-05,2015.0
504759,openalex::W3124716296,Digital Access to Scholarship at Harvard (DASH...,openalex,10.2139/ssrn.2102794,https://doi.org/10.2139/ssrn.2102794,http://nrs.harvard.edu/urn-3:HUL.InstRepos:940...,2012.0,2025-10-10T00:00:00,None,None,2012-01-01,None,None,None,2025-12-10T02:49:46.989445,None,None,None,None,2012-01-01,2012,2102794,2012-07-10,NaT,2012-07-10,2012.0
526424,openalex::W177091947,HAL,openalex,10.2139/ssrn.2201157,https://doi.org/10.2139/ssrn.2201157,https://hal.science/hal-01458342,2014.0,2025-10-10T00:00:00,None,None,2014-01-16,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2014-01-16,2014,2201157,2013-01-22,NaT,2013-01-22,2013.0
692255,openalex::W3125554759,EconStor Preprints,openalex,10.2139/ssrn.3192474,https://doi.org/10.2139/ssrn.3192474,http://hdl.handle.net/10419/271218,2018.0,2021-02-01T00:00:00,None,None,2018-01-01,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2018-01-01,2018,3192474,2018-06-27,NaT,2018-06-27,2018.0
740404,openalex::W2894453084,EconStor Preprints,openalex,10.2139/ssrn.2797203,https://doi.org/10.2139/ssrn.2797203,http://hdl.handle.net/10419/179123,2018.0,2018-10-05T00:00:00,None,None,2018-01-01,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2018-01-01,2018,2797203,2016-06-19,NaT,2016-06-19,2016.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5775710,crossref::10.2139/ssrn.3993882,SSRN,crossref,10.2139/ssrn.3993882,https://doi.org/10.2139/ssrn.3993882,https://www.ssrn.com/abstract=3993882,2021.0,2021-12-28,None,2025-08-01,2021-01-01,None,2021-01-01,2025-08-01,None,None,None,None,None,2021-01-01,2021,3993882,2021-12-27,2021-12-27,2021-12-27,2021.0
5775890,crossref::10.2139/ssrn.4284776,SSRN,crossref,10.2139/ssrn.4284776,https://doi.org/10.2139/ssrn.4284776,https://www.ssrn.com/abstract=4284776,2025.0,2022-12-07,None,2025-08-01,2025-01-01,2025-01-01,2025-01-01,2025-08-01,None,None,None,None,None,2022-12-07,2022,4284776,2022-12-07,2022-12-07,2022-12-07,2022.0
5775933,crossref::10.2139/ssrn.4573444,SSRN,crossref,10.2139/ssrn.4573444,https://doi.org/10.2139/ssrn.4573444,https://www.ssrn.com/abstract=4573444,2023.0,2023-10-11,None,2025-08-01,2023-01-01,2023-01-01,2023-01-01,2025-11-17,None,None,None,None,None,2023-01-01,2023,4573444,2023-10-11,2023-10-11,2023-10-11,2023.0
5776005,crossref::10.2139/ssrn.4344219,SSRN,crossref,10.2139/ssrn.4344219,https://doi.org/10.2139/ssrn.4344219,https://www.ssrn.com/abstract=4344219,2023.0,2023-01-31,None,2025-08-01,2023-01-01,2023-01-01,2023-01-01,2025-12-24,None,None,None,None,None,2023-01-01,2023,4344219,2023-02-01,2023-02-01,2023-02-01,2023.0


In [27]:
date_posted_ssrn_doi['server_name'].value_counts()

server_name
SSRN                                                                    768449
RePEc: Research Papers in Economics                                         14
EconStor Preprints                                                          11
HAL                                                                          1
Digital Access to Scholarship at Harvard (DASH) (Harvard University)         1
Name: count, dtype: int64

In [28]:
date_posted_ssrn_doi.count()

record_id                      768476
server_name                    768476
backend                        768476
doi                            768476
doi_url                        768476
landing_page_url               768476
publication_year               768476
date_created                   768476
date_posted                    125366
date_deposited                 768449
date_published                 768476
date_published_online           49431
date_issued                    768449
date_indexed                   768449
date_updated                       27
date_registered                     0
date_submitted                      0
date_first                          0
raw_dates                           0
date_first_seen                768476
publication_year_first_seen    768476
ssrn_id                        768476
date_posted_ssrn_doi           768476
date_posted_ssrn_url           516456
date_posted_ssrn_id            768476
publication_year_ssrn          768476
dtype: int64

In [29]:
date_posted_ssrn_doi.head(60)

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn
7863,openalex::W3123789860,EconStor Preprints,openalex,10.2139/ssrn.2573596,https://doi.org/10.2139/ssrn.2573596,http://hdl.handle.net/10419/107595,2015.0,2025-10-10T00:00:00,None,None,2015-01-01,None,None,None,2025-10-10T17:16:08.811792,None,None,None,None,2015-01-01,2015,2573596,2015-03-05,NaT,2015-03-05,2015.0
504759,openalex::W3124716296,Digital Access to Scholarship at Harvard (DASH...,openalex,10.2139/ssrn.2102794,https://doi.org/10.2139/ssrn.2102794,http://nrs.harvard.edu/urn-3:HUL.InstRepos:940...,2012.0,2025-10-10T00:00:00,None,None,2012-01-01,None,None,None,2025-12-10T02:49:46.989445,None,None,None,None,2012-01-01,2012,2102794,2012-07-10,NaT,2012-07-10,2012.0
526424,openalex::W177091947,HAL,openalex,10.2139/ssrn.2201157,https://doi.org/10.2139/ssrn.2201157,https://hal.science/hal-01458342,2014.0,2025-10-10T00:00:00,None,None,2014-01-16,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2014-01-16,2014,2201157,2013-01-22,NaT,2013-01-22,2013.0
692255,openalex::W3125554759,EconStor Preprints,openalex,10.2139/ssrn.3192474,https://doi.org/10.2139/ssrn.3192474,http://hdl.handle.net/10419/271218,2018.0,2021-02-01T00:00:00,None,None,2018-01-01,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2018-01-01,2018,3192474,2018-06-27,NaT,2018-06-27,2018.0
740404,openalex::W2894453084,EconStor Preprints,openalex,10.2139/ssrn.2797203,https://doi.org/10.2139/ssrn.2797203,http://hdl.handle.net/10419/179123,2018.0,2018-10-05T00:00:00,None,None,2018-01-01,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2018-01-01,2018,2797203,2016-06-19,NaT,2016-06-19,2016.0
740575,openalex::W3124251451,EconStor Preprints,openalex,10.2139/ssrn.3347766,https://doi.org/10.2139/ssrn.3347766,http://hdl.handle.net/10419/193667,2019.0,2021-02-01T00:00:00,None,None,2019-01-01,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2019-01-01,2019,3347766,2019-04-01,NaT,2019-04-01,2019.0
757851,openalex::W2265447265,EconStor Preprints,openalex,10.2139/ssrn.2458855,https://doi.org/10.2139/ssrn.2458855,http://hdl.handle.net/10419/203278,2015.0,2025-10-10T00:00:00,None,None,2015-01-01,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2015-01-01,2015,2458855,2014-06-27,NaT,2014-06-27,2014.0
765412,openalex::W2296153767,EconStor Preprints,openalex,10.2139/ssrn.2715366,https://doi.org/10.2139/ssrn.2715366,http://hdl.handle.net/10419/146789,2016.0,2025-10-10T00:00:00,None,None,2016-01-01,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2016-01-01,2016,2715366,2016-01-15,NaT,2016-01-15,2016.0
895046,openalex::W3034197722,EconStor Preprints,openalex,10.2139/ssrn.3354400,https://doi.org/10.2139/ssrn.3354400,http://hdl.handle.net/10419/225342,2020.0,2020-06-19T00:00:00,None,None,2020-01-01,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2020-01-01,2020,3354400,2019-04-18,NaT,2019-04-18,2019.0
1308703,openalex::W2792595594,EconStor Preprints,openalex,10.2139/ssrn.3187331,https://doi.org/10.2139/ssrn.3187331,http://hdl.handle.net/10419/179465,2017.0,2025-10-10T00:00:00,None,None,2017-12-01,None,None,None,2025-10-10T17:16:08.811792,None,None,None,None,2017-12-01,2017,3187331,2018-05-30,NaT,2018-05-30,2018.0


In [30]:
date_posted_ssrn_url = data_date_first_seen[data_date_first_seen["date_posted_ssrn_url"].notna()]
date_posted_ssrn_url

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn
2247491,crossref::10.2139/ssrn.3187032,SSRN,crossref,10.2139/ssrn.3187032,https://doi.org/10.2139/ssrn.3187032,https://www.ssrn.com/abstract=3187032,2018.0,2018-06-20,None,2019-05-01,2018-01-01,None,2018-01-01,2022-03-31,None,None,None,None,None,2018-01-01,2018,3187032,2018-06-12,2018-06-12,2018-06-12,2018.0
2247493,crossref::10.2139/ssrn.2673028,SSRN,crossref,10.2139/ssrn.2673028,https://doi.org/10.2139/ssrn.2673028,https://www.ssrn.com/abstract=2673028,2016.0,2015-10-22,None,2019-05-01,2016-01-01,None,2016-01-01,2022-04-02,None,None,None,None,None,2015-10-22,2015,2673028,2015-10-14,2015-10-14,2015-10-14,2015.0
2247495,crossref::10.2139/ssrn.3125251,SSRN,crossref,10.2139/ssrn.3125251,https://doi.org/10.2139/ssrn.3125251,https://www.ssrn.com/abstract=3125251,2017.0,2018-03-07,None,2019-05-01,2017-01-01,None,2017-01-01,2022-04-07,None,None,None,None,None,2017-01-01,2017,3125251,2018-02-27,2018-02-27,2018-02-27,2018.0
2247508,crossref::10.2139/ssrn.2888140,SSRN,crossref,10.2139/ssrn.2888140,https://doi.org/10.2139/ssrn.2888140,https://www.ssrn.com/abstract=2888140,2015.0,2017-06-28,None,2019-05-01,2015-01-01,None,2015-01-01,2022-04-05,None,None,None,None,None,2015-01-01,2015,2888140,2016-12-22,2016-12-22,2016-12-22,2016.0
2247511,crossref::10.2139/ssrn.3167014,SSRN,crossref,10.2139/ssrn.3167014,https://doi.org/10.2139/ssrn.3167014,https://www.ssrn.com/abstract=3167014,2018.0,2018-05-16,None,2019-05-01,2018-01-01,None,2018-01-01,2025-10-05,None,None,None,None,None,2018-01-01,2018,3167014,2018-04-23,2018-04-23,2018-04-23,2018.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5775710,crossref::10.2139/ssrn.3993882,SSRN,crossref,10.2139/ssrn.3993882,https://doi.org/10.2139/ssrn.3993882,https://www.ssrn.com/abstract=3993882,2021.0,2021-12-28,None,2025-08-01,2021-01-01,None,2021-01-01,2025-08-01,None,None,None,None,None,2021-01-01,2021,3993882,2021-12-27,2021-12-27,2021-12-27,2021.0
5775890,crossref::10.2139/ssrn.4284776,SSRN,crossref,10.2139/ssrn.4284776,https://doi.org/10.2139/ssrn.4284776,https://www.ssrn.com/abstract=4284776,2025.0,2022-12-07,None,2025-08-01,2025-01-01,2025-01-01,2025-01-01,2025-08-01,None,None,None,None,None,2022-12-07,2022,4284776,2022-12-07,2022-12-07,2022-12-07,2022.0
5775933,crossref::10.2139/ssrn.4573444,SSRN,crossref,10.2139/ssrn.4573444,https://doi.org/10.2139/ssrn.4573444,https://www.ssrn.com/abstract=4573444,2023.0,2023-10-11,None,2025-08-01,2023-01-01,2023-01-01,2023-01-01,2025-11-17,None,None,None,None,None,2023-01-01,2023,4573444,2023-10-11,2023-10-11,2023-10-11,2023.0
5776005,crossref::10.2139/ssrn.4344219,SSRN,crossref,10.2139/ssrn.4344219,https://doi.org/10.2139/ssrn.4344219,https://www.ssrn.com/abstract=4344219,2023.0,2023-01-31,None,2025-08-01,2023-01-01,2023-01-01,2023-01-01,2025-12-24,None,None,None,None,None,2023-01-01,2023,4344219,2023-02-01,2023-02-01,2023-02-01,2023.0


In [31]:
date_posted_ssrn_url['server_name'].value_counts()

server_name
SSRN    516456
Name: count, dtype: int64

In [32]:
date_posted_ssrn_url.count()

record_id                      516456
server_name                    516456
backend                        516456
doi                            516456
doi_url                        516456
landing_page_url               516456
publication_year               516456
date_created                   516456
date_posted                    125366
date_deposited                 516456
date_published                 516456
date_published_online           49426
date_issued                    516456
date_indexed                   516456
date_updated                        0
date_registered                     0
date_submitted                      0
date_first                          0
raw_dates                           0
date_first_seen                516456
publication_year_first_seen    516456
ssrn_id                        516456
date_posted_ssrn_doi           516456
date_posted_ssrn_url           516456
date_posted_ssrn_id            516456
publication_year_ssrn          516456
dtype: int64

In [33]:
date_posted_ssrn_id = data_date_first_seen[data_date_first_seen["date_posted_ssrn_id"].notna()]
date_posted_ssrn_id

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn
7863,openalex::W3123789860,EconStor Preprints,openalex,10.2139/ssrn.2573596,https://doi.org/10.2139/ssrn.2573596,http://hdl.handle.net/10419/107595,2015.0,2025-10-10T00:00:00,None,None,2015-01-01,None,None,None,2025-10-10T17:16:08.811792,None,None,None,None,2015-01-01,2015,2573596,2015-03-05,NaT,2015-03-05,2015.0
504759,openalex::W3124716296,Digital Access to Scholarship at Harvard (DASH...,openalex,10.2139/ssrn.2102794,https://doi.org/10.2139/ssrn.2102794,http://nrs.harvard.edu/urn-3:HUL.InstRepos:940...,2012.0,2025-10-10T00:00:00,None,None,2012-01-01,None,None,None,2025-12-10T02:49:46.989445,None,None,None,None,2012-01-01,2012,2102794,2012-07-10,NaT,2012-07-10,2012.0
526424,openalex::W177091947,HAL,openalex,10.2139/ssrn.2201157,https://doi.org/10.2139/ssrn.2201157,https://hal.science/hal-01458342,2014.0,2025-10-10T00:00:00,None,None,2014-01-16,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2014-01-16,2014,2201157,2013-01-22,NaT,2013-01-22,2013.0
692255,openalex::W3125554759,EconStor Preprints,openalex,10.2139/ssrn.3192474,https://doi.org/10.2139/ssrn.3192474,http://hdl.handle.net/10419/271218,2018.0,2021-02-01T00:00:00,None,None,2018-01-01,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2018-01-01,2018,3192474,2018-06-27,NaT,2018-06-27,2018.0
740404,openalex::W2894453084,EconStor Preprints,openalex,10.2139/ssrn.2797203,https://doi.org/10.2139/ssrn.2797203,http://hdl.handle.net/10419/179123,2018.0,2018-10-05T00:00:00,None,None,2018-01-01,None,None,None,2025-11-06T03:46:38.306776,None,None,None,None,2018-01-01,2018,2797203,2016-06-19,NaT,2016-06-19,2016.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5775710,crossref::10.2139/ssrn.3993882,SSRN,crossref,10.2139/ssrn.3993882,https://doi.org/10.2139/ssrn.3993882,https://www.ssrn.com/abstract=3993882,2021.0,2021-12-28,None,2025-08-01,2021-01-01,None,2021-01-01,2025-08-01,None,None,None,None,None,2021-01-01,2021,3993882,2021-12-27,2021-12-27,2021-12-27,2021.0
5775890,crossref::10.2139/ssrn.4284776,SSRN,crossref,10.2139/ssrn.4284776,https://doi.org/10.2139/ssrn.4284776,https://www.ssrn.com/abstract=4284776,2025.0,2022-12-07,None,2025-08-01,2025-01-01,2025-01-01,2025-01-01,2025-08-01,None,None,None,None,None,2022-12-07,2022,4284776,2022-12-07,2022-12-07,2022-12-07,2022.0
5775933,crossref::10.2139/ssrn.4573444,SSRN,crossref,10.2139/ssrn.4573444,https://doi.org/10.2139/ssrn.4573444,https://www.ssrn.com/abstract=4573444,2023.0,2023-10-11,None,2025-08-01,2023-01-01,2023-01-01,2023-01-01,2025-11-17,None,None,None,None,None,2023-01-01,2023,4573444,2023-10-11,2023-10-11,2023-10-11,2023.0
5776005,crossref::10.2139/ssrn.4344219,SSRN,crossref,10.2139/ssrn.4344219,https://doi.org/10.2139/ssrn.4344219,https://www.ssrn.com/abstract=4344219,2023.0,2023-01-31,None,2025-08-01,2023-01-01,2023-01-01,2023-01-01,2025-12-24,None,None,None,None,None,2023-01-01,2023,4344219,2023-02-01,2023-02-01,2023-02-01,2023.0


In [34]:
date_posted_ssrn_id['server_name'].value_counts()

server_name
SSRN                                                                    768449
RePEc: Research Papers in Economics                                         14
EconStor Preprints                                                          11
HAL                                                                          1
Digital Access to Scholarship at Harvard (DASH) (Harvard University)         1
Name: count, dtype: int64

In [35]:
date_posted_ssrn_id.count()

record_id                      768476
server_name                    768476
backend                        768476
doi                            768476
doi_url                        768476
landing_page_url               768476
publication_year               768476
date_created                   768476
date_posted                    125366
date_deposited                 768449
date_published                 768476
date_published_online           49431
date_issued                    768449
date_indexed                   768449
date_updated                       27
date_registered                     0
date_submitted                      0
date_first                          0
raw_dates                           0
date_first_seen                768476
publication_year_first_seen    768476
ssrn_id                        768476
date_posted_ssrn_doi           768476
date_posted_ssrn_url           516456
date_posted_ssrn_id            768476
publication_year_ssrn          768476
dtype: int64

In [36]:
date_posted_ssrn_yearna = data_date_first_seen[data_date_first_seen["publication_year_ssrn"].isna()]
date_posted_ssrn_yearna

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn
0,crossref::10.32942/osf.io/brfas,EcoEvoRxiv,crossref,10.32942/osf.io/brfas,https://doi.org/10.32942/osf.io/brfas,https://ecoevorxiv.org/repository/view/3707,2022.0,2022-07-18,2022-07-17,2022-10-25,2022-07-17,None,2022-07-17,2025-10-26,None,None,None,None,None,2022-07-17,2022,NaN,NaT,NaT,NaT,NaN
1,crossref::10.32942/osf.io/gx65w,EcoEvoRxiv,crossref,10.32942/osf.io/gx65w,https://doi.org/10.32942/osf.io/gx65w,https://ecoevorxiv.org/repository/view/3708,2022.0,2022-07-16,2022-07-15,2022-10-25,2022-07-15,None,2022-07-15,2025-02-21,None,None,None,None,None,2022-07-15,2022,NaN,NaT,NaT,NaT,NaN
2,crossref::10.32942/osf.io/b6rf4,EcoEvoRxiv,crossref,10.32942/osf.io/b6rf4,https://doi.org/10.32942/osf.io/b6rf4,https://ecoevorxiv.org/repository/view/3709,2022.0,2022-07-14,2022-07-14,2022-10-25,2022-07-14,None,2022-07-14,2025-10-22,None,None,None,None,None,2022-07-14,2022,NaN,NaT,NaT,NaT,NaN
3,crossref::10.32942/osf.io/fb8k7,EcoEvoRxiv,crossref,10.32942/osf.io/fb8k7,https://doi.org/10.32942/osf.io/fb8k7,https://ecoevorxiv.org/repository/view/3710,2022.0,2022-07-13,2022-07-13,2022-10-25,2022-07-13,None,2022-07-13,2025-02-21,None,None,None,None,None,2022-07-13,2022,NaN,NaT,NaT,NaT,NaN
4,crossref::10.32942/osf.io/sm6xq,EcoEvoRxiv,crossref,10.32942/osf.io/sm6xq,https://doi.org/10.32942/osf.io/sm6xq,https://ecoevorxiv.org/repository/view/3711,2022.0,2022-07-11,2022-07-11,2022-10-25,2022-07-11,None,2022-07-11,2022-10-26,None,None,None,None,None,2022-07-11,2022,NaN,NaT,NaT,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8031087,datacite::10.48550/arxiv.2501.12237,arXiv,datacite,10.48550/arxiv.2501.12237,https://doi.org/10.48550/arxiv.2501.12237,https://arxiv.org/abs/2501.12237,2025.0,2025-01-22,None,None,None,None,None,None,2025-04-11,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T15:59:03Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN
8031088,datacite::10.48550/arxiv.2501.12238,arXiv,datacite,10.48550/arxiv.2501.12238,https://doi.org/10.48550/arxiv.2501.12238,https://arxiv.org/abs/2501.12238,2025.0,2025-01-22,None,None,None,None,None,None,2025-02-27,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T15:59:19Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN
8031089,datacite::10.48550/arxiv.2501.12248,arXiv,datacite,10.48550/arxiv.2501.12248,https://doi.org/10.48550/arxiv.2501.12248,https://arxiv.org/abs/2501.12248,2025.0,2025-01-22,None,None,None,None,None,None,2025-05-08,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T16:09:02Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN
8031090,datacite::10.48550/arxiv.2501.12281,arXiv,datacite,10.48550/arxiv.2501.12281,https://doi.org/10.48550/arxiv.2501.12281,https://arxiv.org/abs/2501.12281,2025.0,2025-01-22,None,None,None,None,None,None,2025-03-21,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T16:52:42Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN


In [37]:
date_posted_ssrn_yearna['publication_year_first_seen'].value_counts()

publication_year_first_seen
2025    983982
2024    740712
2023    527493
2021    497888
2019    493034
2022    491057
2020    479237
2018    306378
2017    265134
2016    225896
2015    208296
2014    194841
2013    184604
2012    172094
2011    158335
2010    146668
2009    134375
2008    123559
2007    116469
2006    107659
2005     98220
2004     84369
2003     74464
2002     65097
2001     57611
2000     54025
1999     48395
1998     43691
1997     37422
1996     32748
1995     28795
1994     24708
1993     19315
1992     15496
1991     10815
1990      9562
2026       123
1989         8
1984         6
1978         4
1987         4
1988         4
1981         3
1982         3
1971         2
1976         2
1983         2
1975         2
1970         2
1828         2
1967         1
1969         1
1961         1
1972         1
1986         1
Name: count, dtype: Int64

### not match in the main file

In [38]:
date_posted_ssrn_yearna_ssrn = date_posted_ssrn_yearna[date_posted_ssrn_yearna['server_name']=='SSRN']
date_posted_ssrn_yearna_ssrn

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn
2247587,crossref::10.2139/ssrn.577262,SSRN,crossref,10.2139/ssrn.577262,https://doi.org/10.2139/ssrn.577262,http://www.ssrn.com/abstract=577262,2004.0,2005-07-07,None,2019-05-01,2004-01-01,None,2004-01-01,2022-04-06,None,None,None,None,None,2004-01-01,2004,577262,NaT,NaT,NaT,NaN
2247604,crossref::10.2139/ssrn.2150161,SSRN,crossref,10.2139/ssrn.2150161,https://doi.org/10.2139/ssrn.2150161,http://www.ssrn.com/abstract=2150161,2006.0,2012-09-27,None,2019-05-01,2006-01-01,None,2006-01-01,2022-03-30,None,None,None,None,None,2006-01-01,2006,2150161,NaT,NaT,NaT,NaN
2247690,crossref::10.2139/ssrn.3157299,SSRN,crossref,10.2139/ssrn.3157299,https://doi.org/10.2139/ssrn.3157299,https://www.ssrn.com/abstract=3157299,2018.0,2018-05-02,None,2019-05-01,2018-01-01,None,2018-01-01,2022-04-01,None,None,None,None,None,2018-01-01,2018,3157299,NaT,NaT,NaT,NaN
2247740,crossref::10.2139/ssrn.3315865,SSRN,crossref,10.2139/ssrn.3315865,https://doi.org/10.2139/ssrn.3315865,https://www.ssrn.com/abstract=3315865,2019.0,2019-05-01,None,2019-05-01,2019-01-01,None,2019-01-01,2022-04-05,None,None,None,None,None,2019-01-01,2019,3315865,NaT,NaT,NaT,NaN
2247776,crossref::10.2139/ssrn.3377270,SSRN,crossref,10.2139/ssrn.3377270,https://doi.org/10.2139/ssrn.3377270,https://www.ssrn.com/abstract=3377270,2019.0,2019-05-01,None,2019-05-01,2019-01-01,None,2019-01-01,2026-03-24,None,None,None,None,None,2019-01-01,2019,3377270,NaT,NaT,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5776184,crossref::10.2139/ssrn.5366340,SSRN,crossref,10.2139/ssrn.5366340,https://doi.org/10.2139/ssrn.5366340,https://www.ssrn.com/abstract=5366340,2025.0,2025-08-01,2025-01-01,2025-08-01,2025-01-01,None,2025-01-01,2025-08-01,None,None,None,None,None,2025-01-01,2025,5366340,NaT,NaT,NaT,NaN
5776185,crossref::10.2139/ssrn.5376066,SSRN,crossref,10.2139/ssrn.5376066,https://doi.org/10.2139/ssrn.5376066,https://www.ssrn.com/abstract=5376066,2025.0,2025-08-01,2025-01-01,2025-08-01,2025-01-01,None,2025-01-01,2025-08-01,None,None,None,None,None,2025-01-01,2025,5376066,NaT,NaT,NaT,NaN
5776186,crossref::10.2139/ssrn.5376068,SSRN,crossref,10.2139/ssrn.5376068,https://doi.org/10.2139/ssrn.5376068,https://www.ssrn.com/abstract=5376068,2025.0,2025-08-01,2025-01-01,2025-08-01,2025-01-01,None,2025-01-01,2025-08-01,None,None,None,None,None,2025-01-01,2025,5376068,NaT,NaT,NaT,NaN
5776187,crossref::10.2139/ssrn.5376087,SSRN,crossref,10.2139/ssrn.5376087,https://doi.org/10.2139/ssrn.5376087,https://www.ssrn.com/abstract=5376087,2025.0,2025-08-01,2025-01-01,2025-08-01,2025-01-01,None,2025-01-01,2025-08-01,None,None,None,None,None,2025-01-01,2025,5376087,NaT,NaT,NaT,NaN


In [39]:
date_posted_ssrn_yearna_ssrn.count()

record_id                      534556
server_name                    534556
backend                        534556
doi                            534556
doi_url                        534556
landing_page_url               534556
publication_year               534556
date_created                   534556
date_posted                    390025
date_deposited                 534556
date_published                 534556
date_published_online           76011
date_issued                    534556
date_indexed                   534556
date_updated                        0
date_registered                     0
date_submitted                      0
date_first                          0
raw_dates                           0
date_first_seen                534556
publication_year_first_seen    534556
ssrn_id                        534556
date_posted_ssrn_doi                0
date_posted_ssrn_url                0
date_posted_ssrn_id                 0
publication_year_ssrn               0
dtype: int64

In [40]:
date_posted_ssrn_yearna_ssrn['publication_year_first_seen'].value_counts()

publication_year_first_seen
2025    262116
2024    187549
2023     15550
2022     15259
2012      5970
2013      4885
2021      4771
2011      4594
2010      3535
2020      3342
2014      3030
2009      2862
2015      2774
2019      2701
2017      2514
2018      2477
2016      2423
2008      2239
2007      1649
2006      1188
2005      1085
2004       542
2003       517
2002       299
2001       208
2000       200
1998        78
1999        73
1997        30
1996        25
1991        11
1995        11
1992         9
1993         9
1994         5
1990         4
1987         4
1981         2
1984         2
1971         2
1983         2
1978         2
1976         2
1967         1
1988         1
1982         1
1972         1
1961         1
1969         1
Name: count, dtype: Int64

In [41]:
date_posted_ssrn_yearna_ssrn['date_created'].value_counts().head(60)

date_created
2011-12-28    8578
2012-01-05    5713
2025-05-06    3332
2025-08-12    2684
2025-05-07    2659
2025-06-13    2652
2024-08-19    2415
2025-06-20    2243
2025-02-24    2126
2025-06-19    1952
2025-07-01    1914
2025-12-12    1813
2025-07-24    1805
2025-09-17    1754
2025-10-29    1687
2025-11-17    1641
2025-10-08    1633
2024-12-13    1577
2025-09-08    1541
2024-09-18    1481
2025-05-28    1475
2025-05-14    1461
2025-01-08    1447
2025-02-25    1439
2025-05-01    1410
2025-05-19    1409
2025-06-23    1385
2025-06-16    1360
2025-06-03    1354
2024-08-22    1338
2025-09-11    1325
2024-12-17    1303
2025-06-24    1298
2025-06-12    1290
2025-03-03    1285
2024-05-07    1282
2025-12-23    1278
2025-12-17    1277
2025-08-28    1274
2025-07-30    1272
2025-12-18    1264
2025-12-03    1261
2025-10-16    1256
2024-05-30    1255
2025-11-18    1233
2025-02-04    1230
2025-01-14    1230
2025-12-22    1223
2024-10-07    1217
2024-04-15    1217
2024-06-04    1206
2025-12-10    1205

## Replace first seen date/year for SSRN records using SSRN posted date

In [42]:
import pandas as pd

def replace_ssrn_first_seen_dates(
    df,
    server_col="server_name",
    server_value="SSRN",
    date_first_seen_col="date_first_seen",
    year_first_seen_col="publication_year_first_seen",
    ssrn_date_col="date_posted_ssrn_doi",
    ssrn_year_col="publication_year_ssrn",
):
    """
    Replace first seen date/year for SSRN records using SSRN posted date
    with safe fallback to existing values.
    """

    df = df.copy()

    # Ensure datetime
    df[date_first_seen_col] = pd.to_datetime(df[date_first_seen_col], errors="coerce")
    df[ssrn_date_col] = pd.to_datetime(df[ssrn_date_col], errors="coerce")

    # Ensure numeric year
    df[year_first_seen_col] = pd.to_numeric(df[year_first_seen_col], errors="coerce")
    df[ssrn_year_col] = pd.to_numeric(df[ssrn_year_col], errors="coerce")

    # Mask for SSRN rows
    mask_ssrn = df[server_col].astype(str).str.upper() == server_value.upper()

    # Mask where SSRN values are valid
    mask_valid_date = mask_ssrn & df[ssrn_date_col].notna()
    mask_valid_year = mask_ssrn & df[ssrn_year_col].notna()

    # Replace date
    df.loc[mask_valid_date, date_first_seen_col] = df.loc[mask_valid_date, ssrn_date_col]

    # Replace year
    df.loc[mask_valid_year, year_first_seen_col] = df.loc[mask_valid_year, ssrn_year_col]

    return df

In [43]:
data_date_first_seen = replace_ssrn_first_seen_dates(data_date_first_seen)
data_date_first_seen["publication_year_first_seen"] = (
    pd.to_datetime(data_date_first_seen["date_first_seen"], errors="coerce")
    .dt.year
)

In [44]:
data_date_first_seen.loc[
    data_date_first_seen["server_name"] == "SSRN",
    ["date_first_seen", "date_posted_ssrn_doi"]
].head(20)

,date_first_seen,date_posted_ssrn_doi
2247485,2011-08-19,2011-08-19
2247486,2014-06-12,2014-06-12
2247487,2010-11-14,2010-11-14
2247488,2005-09-14,2005-09-14
2247489,2014-08-22,2014-08-22
2247490,2016-02-01,2016-02-01
2247491,2018-06-12,2018-06-12
2247492,2013-07-31,2013-07-31
2247493,2015-10-14,2015-10-14
2247494,2016-05-18,2016-05-18


In [45]:
(
    data_date_first_seen["server_name"].eq("SSRN")
    & data_date_first_seen["date_posted_ssrn_doi"].notna()
).sum()

np.int64(768449)

## publication_year is na

In [46]:
data_date_first_seen[data_date_first_seen["publication_year"].isna()]

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn
4275,crossref::10.12688/amrcopenres,AMRC Open Research,crossref,10.12688/amrcopenres,https://doi.org/10.12688/amrcopenres,http://www.amrcopenresearch.org,None,2019-02-19,None,2022-11-18,None,None,None,2022-11-19,None,None,None,None,None,2019-02-19,2019,NaN,NaT,NaT,NaT,NaN
4336,crossref::10.12688/healthopenres,AMRC Open Research,crossref,10.12688/healthopenres,https://doi.org/10.12688/healthopenres,http://www.healthopenresearch.org,None,2022-11-23,None,2025-11-21,None,None,None,2025-11-21,None,None,None,None,None,2022-11-23,2022,NaN,NaT,NaT,NaT,NaN
4340,crossref::10.12688/amrcopenres.crossmark-policy,AMRC Open Research,crossref,10.12688/amrcopenres.crossmark-policy,https://doi.org/10.12688/amrcopenres.crossmark...,https://amrcopenresearch.org/about/policies,None,2018-11-14,None,2018-11-14,None,None,None,2022-04-04,None,None,None,None,None,2018-11-14,2018,NaN,NaT,NaT,NaT,NaN
14331,datacite::10.11588/artdok.00004481,ART-Dok,datacite,10.11588/artdok.00004481,https://doi.org/10.11588/artdok.00004481,http://archiv.ub.uni-heidelberg.de/artdok/id/e...,None,2016-10-14,None,None,None,None,None,None,2018-10-22,2016-10-14,None,None,None,2016-10-14,2016,NaN,NaT,NaT,NaT,NaN
14332,datacite::10.11588/artdok.00004482,ART-Dok,datacite,10.11588/artdok.00004482,https://doi.org/10.11588/artdok.00004482,http://archiv.ub.uni-heidelberg.de/artdok/id/e...,None,2016-10-14,None,None,None,None,None,None,2018-10-22,2016-10-14,None,None,None,2016-10-14,2016,NaN,NaT,NaT,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4764097,crossref::10.21203/rs.3.rs-2909011/v1,Research Square,crossref,10.21203/rs.3.rs-2909011/v1,https://doi.org/10.21203/rs.3.rs-2909011/v1,https://www.researchsquare.com/article/rs-2909...,None,2023-05-08,None,2023-05-08,None,None,None,2025-11-23,None,None,None,None,None,2023-05-08,2023,NaN,NaT,NaT,NaT,NaN
4764098,crossref::10.21203/rs.3.rs-2909026/v1,Research Square,crossref,10.21203/rs.3.rs-2909026/v1,https://doi.org/10.21203/rs.3.rs-2909026/v1,https://www.researchsquare.com/article/rs-2909...,None,2023-05-08,None,2023-05-08,None,None,None,2025-11-23,None,None,None,None,None,2023-05-08,2023,NaN,NaT,NaT,NaT,NaN
4764100,crossref::10.21203/rs.3.rs-2909028/v1,Research Square,crossref,10.21203/rs.3.rs-2909028/v1,https://doi.org/10.21203/rs.3.rs-2909028/v1,https://www.researchsquare.com/article/rs-2909...,None,2023-05-08,None,2023-05-08,None,None,None,2025-11-23,None,None,None,None,None,2023-05-08,2023,NaN,NaT,NaT,NaT,NaN
6816148,crossref::10.7554/elife,eLife,crossref,10.7554/elife,https://doi.org/10.7554/elife,https://elifesciences.org/,None,2017-07-25,None,2017-07-25,None,None,None,2026-04-22,None,None,None,None,None,2017-07-25,2017,NaN,NaT,NaT,NaT,NaN


In [47]:
data_date_first_seen["publication_year"] = (
    data_date_first_seen["publication_year"]
    .replace("", pd.NA)
    .replace(0.0, pd.NA)
    .fillna(data_date_first_seen["publication_year_first_seen"])
)

## publication_year < publication_year_first_seen

In [48]:
data_date_first_seen["publication_year"] = pd.to_numeric(data_date_first_seen["publication_year"], errors="coerce")
data_date_first_seen["publication_year_first_seen"] = pd.to_numeric(data_date_first_seen["publication_year_first_seen"], errors="coerce")

df_inferior_year = data_date_first_seen[
    (data_date_first_seen["publication_year"].notna()) &
    (data_date_first_seen["publication_year_first_seen"].notna()) &
    (data_date_first_seen["publication_year"] < data_date_first_seen["publication_year_first_seen"])
]

df_inferior_year[
    ["record_id", "server_name", "backend",
     "publication_year", "publication_year_first_seen", 'date_first_seen']
]

,record_id,server_name,backend,publication_year,publication_year_first_seen,date_first_seen
10510,datacite::10.11588/artdok.00000934,ART-Dok,datacite,2009.0,2016,2016-09-23
10914,datacite::10.11588/artdok.00003196,ART-Dok,datacite,2015.0,2017,2017-02-06
10938,datacite::10.11588/artdok.00000001,ART-Dok,datacite,2006.0,2017,2017-02-15
10939,datacite::10.11588/artdok.00000003,ART-Dok,datacite,2006.0,2017,2017-02-15
10940,datacite::10.11588/artdok.00000002,ART-Dok,datacite,2006.0,2017,2017-02-15
...,...,...,...,...,...,...
7681164,datacite::10.48550/arxiv.cond-mat/9312097,arXiv,datacite,1993.0,1994,1994-01-02
7681168,datacite::10.48550/arxiv.cond-mat/9312099,arXiv,datacite,1993.0,1994,1994-01-02
7700134,datacite::10.48550/arxiv.hep-ph/9312363,arXiv,datacite,1993.0,1994,1994-01-01
8008529,datacite::10.17613/hs98-7t33,Humanities Commons CORE,datacite,2020.0,2021,2021-11-30


In [49]:
df_inferior_year['backend'].value_counts()

backend
datacite    226651
crossref     95246
Name: count, dtype: int64

In [50]:
df_inferior_year['server_name'].value_counts()

server_name
AgEcon Search              172604
SSRN                        95246
ResearchGate                27484
Humanities Commons CORE     16020
ART-Dok                      4126
PropylaeumDok                2950
AfricArXiv                   1291
Open Science Framework       1063
Zenodo                        730
CrossAsia-Repository          300
arXiv                          49
CERN document server           34
Name: count, dtype: int64

In [51]:
mask = (
    data_date_first_seen["publication_year"].notna() &
    data_date_first_seen["publication_year_first_seen"].notna() &
    (
        data_date_first_seen["publication_year"]
        < data_date_first_seen["publication_year_first_seen"]
    )
)

data_date_first_seen.loc[mask, "date_first_seen"] = (
    data_date_first_seen.loc[mask, "publication_year"]
    .astype(int)
    .astype(str) + "-01-01"
)

data_date_first_seen["date_first_seen"] = pd.to_datetime(
    data_date_first_seen["date_first_seen"],
    errors="coerce"
)

/tmp/ipykernel_39206/3591537641.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['2009-01-01' '2015-01-01' '2006-01-01' ... '1993-01-01' '2020-01-01'
 '2016-01-01']' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  data_date_first_seen.loc[mask, "date_first_seen"] = (


In [52]:
df_inferior_year = data_date_first_seen[
    (data_date_first_seen["publication_year"].notna()) &
    (data_date_first_seen["publication_year_first_seen"].notna()) &
    (data_date_first_seen["publication_year"] < data_date_first_seen["publication_year_first_seen"])
]

df_inferior_year[
    ["record_id", "server_name", "backend",
     "publication_year", "publication_year_first_seen", 'date_first_seen']
]

,record_id,server_name,backend,publication_year,publication_year_first_seen,date_first_seen
10510,datacite::10.11588/artdok.00000934,ART-Dok,datacite,2009.0,2016,2009-01-01
10914,datacite::10.11588/artdok.00003196,ART-Dok,datacite,2015.0,2017,2015-01-01
10938,datacite::10.11588/artdok.00000001,ART-Dok,datacite,2006.0,2017,2006-01-01
10939,datacite::10.11588/artdok.00000003,ART-Dok,datacite,2006.0,2017,2006-01-01
10940,datacite::10.11588/artdok.00000002,ART-Dok,datacite,2006.0,2017,2006-01-01
...,...,...,...,...,...,...
7681164,datacite::10.48550/arxiv.cond-mat/9312097,arXiv,datacite,1993.0,1994,1993-01-01
7681168,datacite::10.48550/arxiv.cond-mat/9312099,arXiv,datacite,1993.0,1994,1993-01-01
7700134,datacite::10.48550/arxiv.hep-ph/9312363,arXiv,datacite,1993.0,1994,1993-01-01
8008529,datacite::10.17613/hs98-7t33,Humanities Commons CORE,datacite,2020.0,2021,2020-01-01


In [53]:
mask = (
    data_date_first_seen["publication_year"].notna() &
    data_date_first_seen["publication_year_first_seen"].notna() &
    (
        data_date_first_seen["publication_year"]
        < data_date_first_seen["publication_year_first_seen"]
    )
)

# replace publication_year
data_date_first_seen.loc[mask, "publication_year_first_seen"] = (
    data_date_first_seen.loc[mask, "publication_year"]
)


In [74]:
data_date_first_seen[data_date_first_seen['record_id']=='datacite::10.11588/artdok.00000934'][
    ["record_id", "server_name", "backend",
     "publication_year", "publication_year_first_seen", 'date_first_seen']
]

,record_id,server_name,backend,publication_year,publication_year_first_seen,date_first_seen
10510,datacite::10.11588/artdok.00000934,ART-Dok,datacite,2009.0,2009,2009-01-01


In [54]:
df_inferior_year = data_date_first_seen[
    (data_date_first_seen["publication_year"].notna()) &
    (data_date_first_seen["publication_year_first_seen"].notna()) &
    (data_date_first_seen["publication_year"] < data_date_first_seen["publication_year_first_seen"])
]

df_inferior_year[
    ["record_id", "server_name", "backend",
     "publication_year", "publication_year_first_seen", 'date_first_seen']
]

,record_id,server_name,backend,publication_year,publication_year_first_seen,date_first_seen


# save

In [55]:
data_date_first_seen[['record_id','server_name','date_first_seen','publication_year','publication_year_first_seen','raw_dates']].to_csv("outputs_new/date_first_seen.csv", index=False)
data_date_first_seen[['record_id','server_name','date_first_seen','publication_year','publication_year_first_seen','raw_dates']].to_pickle("outputs_new/date_first_seen.pkl")

In [56]:
data_date_first_seen[['record_id','date_first_seen','publication_year','publication_year_first_seen','raw_dates']]

,record_id,date_first_seen,publication_year,publication_year_first_seen,raw_dates
0,crossref::10.32942/osf.io/brfas,2022-07-17,2022.0,2022,None
1,crossref::10.32942/osf.io/gx65w,2022-07-15,2022.0,2022,None
2,crossref::10.32942/osf.io/b6rf4,2022-07-14,2022.0,2022,None
3,crossref::10.32942/osf.io/fb8k7,2022-07-13,2022.0,2022,None
4,crossref::10.32942/osf.io/sm6xq,2022-07-11,2022.0,2022,None
...,...,...,...,...,...
8031087,datacite::10.48550/arxiv.2501.12237,2025-01-21,2025.0,2025,"[{""date"": ""2025-01-21T15:59:03Z"", ""dateInforma..."
8031088,datacite::10.48550/arxiv.2501.12238,2025-01-21,2025.0,2025,"[{""date"": ""2025-01-21T15:59:19Z"", ""dateInforma..."
8031089,datacite::10.48550/arxiv.2501.12248,2025-01-21,2025.0,2025,"[{""date"": ""2025-01-21T16:09:02Z"", ""dateInforma..."
8031090,datacite::10.48550/arxiv.2501.12281,2025-01-21,2025.0,2025,"[{""date"": ""2025-01-21T16:52:42Z"", ""dateInforma..."


In [57]:
len(data_date_first_seen["server_name"].value_counts())

109

In [58]:
data_date_first_seen[data_date_first_seen["publication_year"].isna()]

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn


In [59]:
data_date_first_seen['publication_year'].value_counts().head(60)

publication_year
2025.0    976747
2024.0    746420
2023.0    665999
2022.0    595040
2021.0    545458
2020.0    519058
2019.0    394892
2018.0    344730
2017.0    293995
2016.0    260686
2015.0    243313
2014.0    229706
2013.0    218074
2012.0    207330
2011.0    190091
2010.0    175330
2009.0    160430
2008.0    147656
2007.0    135237
2006.0    126403
2005.0    113133
2004.0     97210
2003.0     88736
2002.0     75258
2001.0     67098
2000.0     61527
1999.0     53716
1998.0     48172
1997.0     41028
1996.0     35290
1995.0     31290
1994.0     27012
1993.0     21521
1992.0     17816
1991.0     13374
1990.0     12133
1989.0      2395
1988.0      2233
1987.0      2057
1986.0      1977
1985.0      1841
0.0         1760
1981.0      1715
1984.0      1707
1982.0      1676
1983.0      1598
1980.0      1578
1977.0      1532
1978.0      1514
1979.0      1509
1973.0      1310
1976.0      1264
1975.0      1237
1974.0      1220
1972.0      1001
1969.0       990
1970.0       987
1971.0       9

In [60]:
data_date_first_seen['publication_year'].value_counts().reset_index()['publication_year'][201]

np.float64(1882.0)

In [61]:
# data_date_first_seen[data_date_first_seen['publication_year']<'1990']

In [62]:
# data_date_first_seen[data_date_first_seen['publication_year']<'1990.0']['server_name'].value_counts()

In [63]:
data_date_first_seen.publication_year_first_seen.value_counts().reset_index().sort_values('publication_year_first_seen')

,publication_year_first_seen,count
41,0,1760
158,18,1
116,106,8
125,212,4
146,220,2
...,...,...
3,2022,595327
2,2023,667385
1,2024,746387
0,2025,974173


In [64]:
data_date_first_seen.sort_values('publication_year_first_seen')

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn
1677821,datacite::10.22004/ag.econ.352582,AgEcon Search,datacite,10.22004/ag.econ.352582,https://doi.org/10.22004/ag.econ.352582,https://ageconsearch.umn.edu/record/352582,0.0,2025-03-18,None,None,None,None,None,None,2025-03-18,2025-03-18,None,0000,"[{""date"": ""0000"", ""dateType"": ""Issued""}]",NaT,0,NaN,NaT,NaT,NaT,NaN
1568328,datacite::10.22004/ag.econ.342662,AgEcon Search,datacite,10.22004/ag.econ.342662,https://doi.org/10.22004/ag.econ.342662,https://ageconsearch.umn.edu/record/342662,0.0,2024-05-21,None,None,None,None,None,None,2024-05-21,2024-05-21,None,0000,"[{""date"": ""0000"", ""dateType"": ""Issued""}]",NaT,0,NaN,NaT,NaT,NaT,NaN
1568327,datacite::10.22004/ag.econ.342661,AgEcon Search,datacite,10.22004/ag.econ.342661,https://doi.org/10.22004/ag.econ.342661,https://ageconsearch.umn.edu/record/342661,0.0,2024-05-21,None,None,None,None,None,None,2024-05-21,2024-05-21,None,0000,"[{""date"": ""0000"", ""dateType"": ""Issued""}]",NaT,0,NaN,NaT,NaT,NaT,NaN
1677637,datacite::10.22004/ag.econ.352398,AgEcon Search,datacite,10.22004/ag.econ.352398,https://doi.org/10.22004/ag.econ.352398,https://ageconsearch.umn.edu/record/352398,0.0,2025-03-18,None,None,None,None,None,None,2025-03-18,2025-03-18,None,0000,"[{""date"": ""0000"", ""dateType"": ""Issued""}]",NaT,0,NaN,NaT,NaT,NaT,NaN
1677829,datacite::10.22004/ag.econ.352590,AgEcon Search,datacite,10.22004/ag.econ.352590,https://doi.org/10.22004/ag.econ.352590,https://ageconsearch.umn.edu/record/352590,0.0,2025-03-18,None,None,None,None,None,None,2025-03-18,2025-03-18,None,0000,"[{""date"": ""0000"", ""dateType"": ""Issued""}]",NaT,0,NaN,NaT,NaT,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819271,jxiv::10.51094/jxiv.2643,Jxiv,jxiv,10.51094/jxiv.2643,https://doi.org/10.51094/jxiv.2643,https://jxiv.jst.go.jp/index.php/jxiv/preprint...,2026.0,None,2026-01-28,None,None,None,None,None,None,None,None,None,None,2026-01-28,2026,NaN,NaT,NaT,NaT,NaN
819272,jxiv::10.51094/jxiv.2661,Jxiv,jxiv,10.51094/jxiv.2661,https://doi.org/10.51094/jxiv.2661,https://jxiv.jst.go.jp/index.php/jxiv/preprint...,2026.0,None,2026-02-05,None,None,None,None,None,None,None,None,None,None,2026-02-05,2026,NaN,NaT,NaT,NaT,NaN
819273,jxiv::10.51094/jxiv.2664,Jxiv,jxiv,10.51094/jxiv.2664,https://doi.org/10.51094/jxiv.2664,https://jxiv.jst.go.jp/index.php/jxiv/preprint...,2026.0,None,2026-03-04,None,None,None,None,None,None,None,None,None,None,2026-03-04,2026,NaN,NaT,NaT,NaT,NaN
819274,jxiv::10.51094/jxiv.2667,Jxiv,jxiv,10.51094/jxiv.2667,https://doi.org/10.51094/jxiv.2667,https://jxiv.jst.go.jp/index.php/jxiv/preprint...,2026.0,None,2026-01-21,None,None,None,None,None,None,None,None,None,None,2026-01-21,2026,NaN,NaT,NaT,NaT,NaN


# arxiv_data

In [65]:
arxiv_data = data_date_first_seen[data_date_first_seen['server_name']=='arXiv']
arxiv_data

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn
3783485,datacite::10.48550/arxiv.2104.03534,arXiv,datacite,10.48550/arxiv.2104.03534,https://doi.org/10.48550/arxiv.2104.03534,https://arxiv.org/abs/2104.03534,2021.0,2022-02-21,None,None,None,None,None,None,2022-02-21,2022-02-21,2021-04-08,2021-04-08,"[{""date"": ""2021-04-08T06:39:08Z"", ""dateInforma...",2021-04-08,2021,NaN,NaT,NaT,NaT,NaN
3783486,datacite::10.48550/arxiv.2107.07196,arXiv,datacite,10.48550/arxiv.2107.07196,https://doi.org/10.48550/arxiv.2107.07196,https://arxiv.org/abs/2107.07196,2021.0,2022-02-21,None,None,None,None,None,None,2022-02-21,2022-02-21,2021-07-15,2021-07-15,"[{""date"": ""2021-07-15T08:52:38Z"", ""dateInforma...",2021-07-15,2021,NaN,NaT,NaT,NaT,NaN
3783487,datacite::10.48550/arxiv.2108.11460,arXiv,datacite,10.48550/arxiv.2108.11460,https://doi.org/10.48550/arxiv.2108.11460,https://arxiv.org/abs/2108.11460,2021.0,2022-02-21,None,None,None,None,None,None,2022-02-21,2022-02-21,2021-08-25,2021-08-25,"[{""date"": ""2021-08-25T20:20:57Z"", ""dateInforma...",2021-08-25,2021,NaN,NaT,NaT,NaT,NaN
3783488,datacite::10.48550/arxiv.2107.04603,arXiv,datacite,10.48550/arxiv.2107.04603,https://doi.org/10.48550/arxiv.2107.04603,https://arxiv.org/abs/2107.04603,2021.0,2022-02-21,None,None,None,None,None,None,2022-02-21,2022-02-21,2021-07-09,2021-07-09,"[{""date"": ""2021-07-09T18:00:05Z"", ""dateInforma...",2021-07-09,2021,NaN,NaT,NaT,NaT,NaN
3783489,datacite::10.48550/arxiv.2105.03496,arXiv,datacite,10.48550/arxiv.2105.03496,https://doi.org/10.48550/arxiv.2105.03496,https://arxiv.org/abs/2105.03496,2021.0,2022-02-21,None,None,None,None,None,None,2022-02-21,2022-02-21,2021-05-07,2021-05-07,"[{""date"": ""2021-05-07T20:30:47Z"", ""dateInforma...",2021-05-07,2021,NaN,NaT,NaT,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8031087,datacite::10.48550/arxiv.2501.12237,arXiv,datacite,10.48550/arxiv.2501.12237,https://doi.org/10.48550/arxiv.2501.12237,https://arxiv.org/abs/2501.12237,2025.0,2025-01-22,None,None,None,None,None,None,2025-04-11,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T15:59:03Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN
8031088,datacite::10.48550/arxiv.2501.12238,arXiv,datacite,10.48550/arxiv.2501.12238,https://doi.org/10.48550/arxiv.2501.12238,https://arxiv.org/abs/2501.12238,2025.0,2025-01-22,None,None,None,None,None,None,2025-02-27,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T15:59:19Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN
8031089,datacite::10.48550/arxiv.2501.12248,arXiv,datacite,10.48550/arxiv.2501.12248,https://doi.org/10.48550/arxiv.2501.12248,https://arxiv.org/abs/2501.12248,2025.0,2025-01-22,None,None,None,None,None,None,2025-05-08,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T16:09:02Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN
8031090,datacite::10.48550/arxiv.2501.12281,arXiv,datacite,10.48550/arxiv.2501.12281,https://doi.org/10.48550/arxiv.2501.12281,https://arxiv.org/abs/2501.12281,2025.0,2025-01-22,None,None,None,None,None,None,2025-03-21,2025-01-22,2025-01-21,2025-01-21,"[{""date"": ""2025-01-21T16:52:42Z"", ""dateInforma...",2025-01-21,2025,NaN,NaT,NaT,NaT,NaN


In [66]:
arxiv_data[arxiv_data['date_submitted'].isna()]

,record_id,server_name,backend,doi,doi_url,landing_page_url,publication_year,date_created,date_posted,date_deposited,date_published,date_published_online,date_issued,date_indexed,date_updated,date_registered,date_submitted,date_first,raw_dates,date_first_seen,publication_year_first_seen,ssrn_id,date_posted_ssrn_doi,date_posted_ssrn_url,date_posted_ssrn_id,publication_year_ssrn
3793169,datacite::10.48550/arxiv.2103.00799,arXiv,datacite,10.48550/arxiv.2103.00799,https://doi.org/10.48550/arxiv.2103.00799,https://arxiv.org/abs/2103.00799,2021.0,2022-02-22,None,None,None,None,None,None,2022-02-22,2022-02-22,None,2021-03-01,"[{""date"": ""2021-03-01T06:43:43Z"", ""dateInforma...",2021-03-01,2021,NaN,NaT,NaT,NaT,NaN
3960286,datacite::10.48550/arxiv.2005.00229,arXiv,datacite,10.48550/arxiv.2005.00229,https://doi.org/10.48550/arxiv.2005.00229,https://arxiv.org/abs/2005.00229,2020.0,2022-02-25,None,None,None,None,None,None,2022-02-25,2022-02-25,None,2020-05-01,"[{""date"": ""2020-05-01T05:27:16Z"", ""dateInforma...",2020-05-01,2020,NaN,NaT,NaT,NaT,NaN
4081101,datacite::10.48550/arxiv.2110.13637,arXiv,datacite,10.48550/arxiv.2110.13637,https://doi.org/10.48550/arxiv.2110.13637,https://arxiv.org/abs/2110.13637,2021.0,2022-02-21,None,None,None,None,None,None,2022-07-05,2022-02-21,None,2021-10-26,"[{""date"": ""2021-10-26T12:38:36Z"", ""dateInforma...",2021-10-26,2021,NaN,NaT,NaT,NaT,NaN
4198574,datacite::10.48550/arxiv.2007.04303,arXiv,datacite,10.48550/arxiv.2007.04303,https://doi.org/10.48550/arxiv.2007.04303,https://arxiv.org/abs/2007.04303,2020.0,2022-02-24,None,None,None,None,None,None,2022-02-24,2022-02-24,None,2020-07-05,"[{""date"": ""2020-07-05T08:10:30Z"", ""dateInforma...",2020-07-05,2020,NaN,NaT,NaT,NaT,NaN
4198665,datacite::10.48550/arxiv.2006.04345,arXiv,datacite,10.48550/arxiv.2006.04345,https://doi.org/10.48550/arxiv.2006.04345,https://arxiv.org/abs/2006.04345,2020.0,2022-02-24,None,None,None,None,None,None,2022-02-24,2022-02-24,None,2020-06-08,"[{""date"": ""2020-06-08T04:07:57Z"", ""dateInforma...",2020-06-08,2020,NaN,NaT,NaT,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7904050,datacite::10.48550/arxiv.2508.00588,arXiv,datacite,10.48550/arxiv.2508.00588,https://doi.org/10.48550/arxiv.2508.00588,https://arxiv.org/abs/2508.00588,2025.0,2025-08-04,None,None,None,None,None,None,2025-09-04,2025-08-04,None,2025-08-01,"[{""date"": ""2025-08-01T12:41:40Z"", ""dateInforma...",2025-08-01,2025,NaN,NaT,NaT,NaT,NaN
7905808,datacite::10.48550/arxiv.2405.07425,arXiv,datacite,10.48550/arxiv.2405.07425,https://doi.org/10.48550/arxiv.2405.07425,https://arxiv.org/abs/2405.07425,2024.0,2024-05-14,None,None,None,None,None,None,2024-05-24,2024-05-14,None,2024-05-13,"[{""date"": ""2024-05-13T01:50:05Z"", ""dateInforma...",2024-05-13,2024,NaN,NaT,NaT,NaT,NaN
7970255,datacite::10.48550/arxiv.2512.09237,arXiv,datacite,10.48550/arxiv.2512.09237,https://doi.org/10.48550/arxiv.2512.09237,https://arxiv.org/abs/2512.09237,2025.0,2025-12-11,None,None,None,None,None,None,2025-12-15,2025-12-11,None,2025-12-10,"[{""date"": ""2025-12-10T01:54:43Z"", ""dateInforma...",2025-12-10,2025,NaN,NaT,NaT,NaT,NaN
7973443,datacite::10.48550/arxiv.2507.20322,arXiv,datacite,10.48550/arxiv.2507.20322,https://doi.org/10.48550/arxiv.2507.20322,https://arxiv.org/abs/2507.20322,2025.0,2025-08-21,None,None,None,None,None,None,2025-12-01,2025-08-21,None,2025-07-27,"[{""date"": ""2025-07-27T15:22:39Z"", ""dateInforma...",2025-07-27,2025,NaN,NaT,NaT,NaT,NaN


In [67]:
# arxiv_data['date_submitted'][3787581]

In [68]:
# arxiv_data['date_first'][3787581]

In [69]:
arxiv_data['publication_year'].value_counts()

publication_year
2025.0    283120
2024.0    244010
2023.0    208493
2022.0    185688
2021.0    181630
2020.0    178328
2019.0    155866
2018.0    140616
2017.0    123523
2016.0    113380
2015.0    105280
2014.0     97517
2013.0     92641
2012.0     84603
2011.0     76574
2010.0     70125
2009.0     64046
2008.0     58915
2007.0     55638
2006.0     50224
2005.0     46827
2004.0     43708
2003.0     39409
2002.0     36102
2001.0     33197
2000.0     30587
1999.0     27695
1998.0     24167
1997.0     19624
1996.0     15866
1995.0     13002
1994.0     10088
1993.0      6741
1992.0      3261
1991.0       306
Name: count, dtype: int64

In [70]:
arxiv_data['publication_year_first_seen'].value_counts()

publication_year_first_seen
2025    282101
2024    243671
2023    209224
2022    185983
2021    181599
2020    178274
2019    155917
2018    140377
2017    123781
2016    113440
2015    105130
2014     97590
2013     92875
2012     84374
2011     76602
2010     70288
2009     64071
2008     58810
2007     55749
2006     50303
2005     46875
2004     43714
2003     39392
2002     36104
2001     33135
2000     30657
1999     27696
1998     24170
1997     19614
1996     15874
1995     13005
1994     10079
1993      6746
1992      3186
1991       357
1990        26
1989         6
1988         1
1986         1
Name: count, dtype: int64

In [71]:
arxiv_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2920797 entries, 3783485 to 8031091
Data columns (total 26 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   record_id                    object        
 1   server_name                  object        
 2   backend                      object        
 3   doi                          object        
 4   doi_url                      object        
 5   landing_page_url             object        
 6   publication_year             float64       
 7   date_created                 object        
 8   date_posted                  object        
 9   date_deposited               object        
 10  date_published               object        
 11  date_published_online        object        
 12  date_issued                  object        
 13  date_indexed                 object        
 14  date_updated                 object        
 15  date_registered              object        
 16 